[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C53_RealTime_Detectors_Course/01_yolo_evolution/01_yolo_evolution.ipynb)

# 01 · YOLO 家族演进（网格容量 / anchor 聚类 / 重参数化 / 解耦头账 / 双分配）

这个 notebook 不训练模型，它把 YOLO 演进史上**四个关键机制**从零实现出来，
并且每一个都用 `assert` 把「它到底带来了什么」钉死成数字。

**本 notebook 你会亲手实现：**
1. **YOLOv1 的结构性容量上限** —— 证明它的失败是输出张量形状决定的，不是参数量不够
2. **anchor 维度聚类（k-means with 1−IoU 距离）** —— 并证明 IoU 距离优于欧氏距离、
   COCO 默认 anchor 用在交通标志上是灾难
3. **Conv+BN 融合与 RepVGG 三分支合并** —— 完整的**数值等价证明**（误差 < 1e-14）
4. **解耦头 vs 耦合头的参数量/FLOPs 账** —— 那个 37.5× 是怎么来的、怎么压下去
5. **一对多 vs 一对一分配** —— 复现 YOLOv10 「一致双分配」为什么必须"一致"

> 心智模型：**凡是只在训练期付出代价的改动，都是免费的午餐。
> 所以看到一个新机制，先问它的代价落在训练期还是推理期。**

## 1 · YOLOv1 的容量上限是**结构性**的

v1 每个格子输出 `B` 个框 + **一份**类别向量。这两条限制在参数量增加时一条都不会消失。

用一个合成的 TSR 场景来量化：一个施工区门架上 5 块牌子挤在 100 px 内，路侧另有 4 块零散标志。

In [ ]:
import numpy as np, math, json, itertools
rng = np.random.default_rng(53)

def yolov1_capacity(gt_boxes, gt_cls, img=448, S=7, B=2):
    '''YOLOv1: SxS 网格，每格 B 个框、**一份**类别向量（先到先得）。
       返回 (可正确表达的目标数, 因每格超过 B 个框而丢失, 因每格只有一份类别向量而丢失)。'''
    cell = img / S
    buckets = {}
    for (cx, cy, w, h), c in zip(gt_boxes, gt_cls):
        key = (int(min(S - 1, cx // cell)), int(min(S - 1, cy // cell)))
        buckets.setdefault(key, []).append(c)
    ok = lost_box = lost_cls = 0
    for key, cs in sorted(buckets.items()):
        keep = min(len(cs), B)
        lost_box += len(cs) - keep          # (1) 每格最多 B 个框
        main = cs[0]                        # (2) 每格只有一份类别向量
        same = sum(1 for c in cs[:keep] if c == main)
        lost_cls += keep - same
        ok += same
    return ok, lost_box, lost_cls, buckets

# 合成场景：门架 5 块牌（类别 0/1/0/2/0）+ 路侧 4 块零散标志
gt, cls = [], []
for i in range(5):
    gt.append((150 + i * 22, 120, 18, 18)); cls.append([0, 1, 0, 2, 0][i])
for i in range(4):
    gt.append((60 + i * 95, 260 + (i % 2) * 30, 24, 24)); cls.append(i % 3)

ok, lb, lc, buckets = yolov1_capacity(gt, cls)
print('场景里共 %d 个交通标志，448 输入 / 7x7 网格（每格 %.0f px）' % (len(gt), 448 / 7))
print('\n每个被占用的格子里挤了几个标志：')
for key, cs in sorted(buckets.items()):
    flag = '   <-- 超过 B=2' if len(cs) > 2 else ('   <-- 两个不同类别，只能保住一个'
                                                  if len(set(cs)) > 1 else '')
    print('   格子 %s  %d 个  类别 %s%s' % (str(key), len(cs), cs, flag))

print('\nYOLOv1 结构上最多能表达: %d / %d' % (ok, len(gt)))
print('   因「每格 B=2」丢失          : %d' % lb)
print('   因「每格一份类别向量」丢失  : %d' % lc)
assert (ok, lb, lc) == (6, 1, 2), (ok, lb, lc)
assert ok + lb + lc == len(gt)

In [ ]:
def dense_capacity(gt_boxes, img=640, stride=8, n_pred=3, src=448):
    '''现代 dense head：stride 8 的格点，每格 n_pred 个预测，且**每个预测自带类别**。'''
    scale = img / src
    b = {}
    for (cx, cy, w, h) in gt_boxes:
        k = (int(cx * scale // stride), int(cy * scale // stride))
        b[k] = b.get(k, 0) + 1
    lost = sum(max(0, v - n_pred) for v in b.values())
    return len(gt_boxes) - lost, lost

got, lost = dense_capacity(gt)
print('同一批标志交给 stride 8 的 dense head（640 输入 -> 80x80 格点）:')
print('   可表达 %d / %d，丢失 %d' % (got, len(gt), lost))
assert (got, lost) == (9, 0)

print('\n%-34s %10s %12s' % ('结构', '格点数', '可表达上限'))
for name, S, B in [('YOLOv1  7x7, B=2, 每格 1 个类别', 7, 2),
                   ('YOLOv2  13x13, 5 anchor, 各自带类别', 13, 5),
                   ('YOLOv3+ stride8 80x80, 3 anchor', 80, 3)]:
    print('%-34s %10d %12d' % (name, S * S, S * S * B))

print('\n⚠️  v1 的失败是**表达能力**问题不是**容量**问题 —— 加参数一条都解决不了。')
print('✅ 可迁移的判断法：模型在某类样本上**系统性**失败时，先问')
print('   「这是容量问题（加参数能解决）还是表达能力问题（必须改输出结构）」。')

## 2 · anchor 维度聚类：k-means with 1−IoU 距离

YOLOv2 的 dimension cluster。**距离必须用 1−IoU 而不是欧氏距离**，
因为欧氏距离会被大框主导：300×300 vs 320×320 的欧氏距离是 28、IoU 0.88；
8×8 vs 10×10 的欧氏距离只有 2.8、IoU 却只有 0.64。

数据用**针孔模型合成的交通标志尺寸分布**：`px = f·S/Z`，
物理尺寸 0.6/0.8/1.2 m，距离 10–120 m，1920 宽的图缩放到 1280 的网络输入。

In [ ]:
def make_tsr_wh(n=3000, seed=7, img_w=1920, net_w=1280, hfov=60.0):
    '''按针孔相机模型合成交通标志的 (w,h) 分布 —— 物理上可解释、可复现。'''
    r = np.random.default_rng(seed)
    f = (img_w / 2) / math.tan(math.radians(hfov) / 2)      # 焦距（像素）
    size_m = r.choice([0.6, 0.8, 1.2], size=n, p=[0.60, 0.25, 0.15])   # 标志物理尺寸
    dist_m = r.uniform(10.0, 120.0, size=n)                            # 距离
    w = f * size_m / dist_m * (net_w / img_w) * r.normal(1.0, 0.08, n)  # 含标注/姿态噪声
    h = w * r.normal(1.0, 0.10, n)                                     # 长宽比 ~1
    return np.stack([np.abs(w), np.abs(h)], 1)

def wh_iou(wh, centers):
    '''只比 (w,h) 的 IoU（两框中心对齐）。wh:(n,2) centers:(k,2) -> (n,k)'''
    inter = (np.minimum(wh[:, None, 0], centers[None, :, 0])
             * np.minimum(wh[:, None, 1], centers[None, :, 1]))
    union = wh[:, None, 0] * wh[:, None, 1] + centers[None, :, 0] * centers[None, :, 1] - inter
    return inter / union

def kmeans_anchors(wh, k, iters=100, seed=0, metric='iou'):
    r = np.random.default_rng(seed)
    c = wh[r.choice(len(wh), k, replace=False)].astype(float).copy()
    for _ in range(iters):
        d = (1.0 - wh_iou(wh, c)) if metric == 'iou' else \
            np.linalg.norm(wh[:, None, :] - c[None, :, :], axis=2)
        a = d.argmin(1)
        new = c.copy()
        for j in range(k):
            m = a == j
            if m.any():
                new[j] = wh[m].mean(0)
        if np.allclose(new, c):
            break
        c = new
    return c[np.argsort(c[:, 0] * c[:, 1])]            # 按面积排序，便于分配到 FPN 层

def mean_best_iou(wh, c):
    return float(wh_iou(wh, c).max(1).mean())

def bpr(wh, c, thr=0.25):
    '''best possible recall: 有多少比例的框能找到 IoU>=thr 的 anchor。'''
    return float((wh_iou(wh, c).max(1) >= thr).mean())

wh = make_tsr_wh()
print('合成的 TSR 尺寸分布（网络输入 1280 尺度下）:')
print('   最小 %.1f px   中位 %.1f px   最大 %.1f px'
      % (wh.min(), np.median(np.sqrt(wh[:, 0] * wh[:, 1])), wh.max()))
print('\n%6s %18s %12s' % ('k', '平均最佳 IoU', 'BPR@0.25'))
res = {}
for k in (1, 3, 6, 9, 12):
    a = kmeans_anchors(wh, k)
    res[k] = (mean_best_iou(wh, a), bpr(wh, a))
    print('%6d %18.4f %12.4f' % (k, res[k][0], res[k][1]))
assert res[1][0] < res[3][0] < res[6][0] < res[9][0] < res[12][0], 'anchor 越多拟合越好'
assert res[9][1] == 1.0 and res[3][1] < 1.0
print('\n✅ k=9 时平均最佳 IoU %.4f、BPR=1.0（每个标志都能找到 IoU>=0.25 的 anchor）' % res[9][0])

In [ ]:
# ① IoU 距离 vs 欧氏距离
print('%6s %16s %16s   %s' % ('k', 'IoU 距离', '欧氏距离', '差值'))
for k in (3, 6, 9):
    ai = kmeans_anchors(wh, k, metric='iou')
    ae = kmeans_anchors(wh, k, metric='euclid')
    mi, me = mean_best_iou(wh, ai), mean_best_iou(wh, ae)
    print('%6d %16.4f %16.4f   %+.4f' % (k, mi, me, mi - me))
    assert mi > me, 'IoU 距离在 k=%d 上应优于欧氏距离' % k
print('\n>>> 欧氏距离被大框主导，聚类中心被拽走，小目标分不到 anchor。')

# ② 直接用 COCO 的默认 anchor 会怎样
COCO_ANCHORS = np.array([[10, 13], [16, 30], [33, 23], [30, 61], [62, 45],
                         [59, 119], [116, 90], [156, 198], [373, 326]], float)
a9 = kmeans_anchors(wh, 9)

def at_smallest(wh, a):
    '''有多少比例的框，其最佳 anchor 是**最小的那一个** -> anchor 下界不够的信号。'''
    return float((wh_iou(wh, a).argmax(1) == int(np.argmin(a[:, 0] * a[:, 1]))).mean())

print('\n%-26s %14s %10s %16s' % ('anchor 来源', '平均最佳 IoU', 'BPR@0.25', '挤在最小 anchor'))
for name, a in [('COCO 默认', COCO_ANCHORS), ('在 TSR 上重新聚类', a9)]:
    print('%-26s %14.4f %10.4f %15.1f%%'
          % (name, mean_best_iou(wh, a), bpr(wh, a), 100 * at_smallest(wh, a)))

assert bpr(wh, COCO_ANCHORS) < 0.97 and bpr(wh, a9) == 1.0
assert mean_best_iou(wh, a9) > mean_best_iou(wh, COCO_ANCHORS) + 0.2
assert at_smallest(wh, COCO_ANCHORS) > 0.5 > at_smallest(wh, a9)

print('\n重新聚类得到的 9 个 anchor:')
print('  ', [[round(x, 1) for x in p] for p in a9.tolist()])
print('\n⚠️  COCO 默认 anchor 让 %.1f%% 的标志在任何 IoU 阈值下都匹配不上，'
      % (100 * (1 - bpr(wh, COCO_ANCHORS))))
print('    且 %.1f%% 的标志全挤在最小的 [10,13] 上 —— 这是「anchor 下界不够」的明确信号。'
      % (100 * at_smallest(wh, COCO_ANCHORS)))
print('✅ 注意聚出来的 anchor 长宽比几乎全是 1 —— **TSR 的 anchor 只在编码尺度先验**，')
print('   完全没在编码长宽比先验。这正是 anchor-free 在 TSR 上代价极小的原因。')

In [ ]:
# ③ 多尺度：这份分布在三层 FPN 上的负载
side = np.sqrt(wh[:, 0] * wh[:, 1])
print('%-22s %10s %10s' % ('FPN 层', '样本数', '占比'))
loads = []
for lo, hi, name in [(0, 32, 'P3 (stride 8)'), (32, 64, 'P4 (stride 16)'),
                     (64, 1e9, 'P5 (stride 32)')]:
    m = (side >= lo) & (side < hi)
    loads.append(m.sum())
    print('%-22s %10d %9.1f%%' % (name, m.sum(), 100 * m.mean()))

tiny = (side < 8).sum()
assert loads[0] > 0.8 * len(wh), 'TSR 的绝大多数目标都落在最细的那一层'
assert loads[2] < 0.05 * len(wh), 'P5 几乎没有负载'
assert tiny > 0.2 * len(wh)

print('\n>>> %.1f%% 落在 P3，只有 %.1f%% 落在 P5 —— **P5 的算力几乎白花**。'
      % (100 * loads[0] / len(wh), 100 * loads[2] / len(wh)))
print('>>> 更刺眼的是：%.1f%% 的标志边长 < 8 px，在 stride 8 上**连一个格子都占不满**。'
      % (100 * tiny / len(wh)))
print('⚠️  对这批样本，无论标签分配怎么改、损失怎么调，信息在下采样时就已经丢了。')
print('✅ 这是架构层面的天花板 —— 只能靠更细的 stride(P2)、更高输入分辨率或切片推理解决(C57)。')

## 3 · Conv+BN 融合：最简单的一次结构重参数化

推理期的 BatchNorm 是一个逐通道的仿射变换，可以被**严格等价地**吸进前面卷积的权重与偏置：

$$W' = \frac{\gamma}{\sqrt{\sigma^2+\epsilon}} \odot W, \qquad
b' = \frac{\gamma\,(b-\mu)}{\sqrt{\sigma^2+\epsilon}} + \beta$$

先自己写一个 numpy conv2d，然后把这个等式用 `assert` 钉死。

In [ ]:
def conv2d(x, W, b=None, stride=1, pad=0):
    '''x:(N,Cin,H,W)  W:(Cout,Cin,kh,kw)  -> (N,Cout,Ho,Wo)。零填充。'''
    N, Cin, H, Wd = x.shape
    Cout, Cin2, kh, kw = W.shape
    assert Cin == Cin2, '输入通道不匹配'
    xp = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    Ho = (H + 2 * pad - kh) // stride + 1
    Wo = (Wd + 2 * pad - kw) // stride + 1
    out = np.zeros((N, Cout, Ho, Wo))
    for i in range(kh):                       # 按核内位置累加，避免显式 im2col
        for j in range(kw):
            patch = xp[:, :, i:i + stride * Ho:stride, j:j + stride * Wo:stride]
            out += np.einsum('nchw,oc->nohw', patch, W[:, :, i, j])
    if b is not None:
        out += b.reshape(1, -1, 1, 1)
    return out

def bn_infer(x, gamma, beta, mean, var, eps=1e-5):
    '''**推理期**的 BN：用 running statistics，是一个逐通道仿射变换。'''
    r = lambda v: v.reshape(1, -1, 1, 1)
    return r(gamma) * (x - r(mean)) / np.sqrt(r(var) + eps) + r(beta)

def fuse_conv_bn(W, b, gamma, beta, mean, var, eps=1e-5):
    '''把 BN 吸进卷积。返回等价的 (W', b')。'''
    s = gamma / np.sqrt(var + eps)
    return W * s.reshape(-1, 1, 1, 1), (b - mean) * s + beta

C, H = 8, 7
x = rng.normal(size=(2, C, H, H))
W3 = rng.normal(size=(C, C, 3, 3)) * 0.2
b3 = rng.normal(size=C) * 0.1
g3, be3 = rng.normal(1, .2, C), rng.normal(0, .2, C)
mu3, v3 = rng.normal(0, .3, C), rng.uniform(.5, 2., C)      # var 必须为正

y_two_step = bn_infer(conv2d(x, W3, b3, pad=1), g3, be3, mu3, v3)   # conv 然后 bn
Wf, bf = fuse_conv_bn(W3, b3, g3, be3, mu3, v3)
y_fused = conv2d(x, Wf, bf, pad=1)                                  # 融合后的单个 conv

err = float(np.abs(y_two_step - y_fused).max())
print('Conv->BN 两步   输出形状 %s' % (y_two_step.shape,))
print('融合成单个 conv 输出形状 %s' % (y_fused.shape,))
print('最大绝对误差 %.3e   （float64 机器精度量级）' % err)
assert np.allclose(y_two_step, y_fused, atol=1e-10), err
assert err < 1e-12
print('\n✅ **严格数值等价**，不是近似。推理时 BN 层可以整个消失：')
print('   少一次逐元素读写、少一个 kernel 启动，且不再阻碍算子融合。')

## 4 · RepVGG：三分支合并成一个 3×3

训练时三条并行分支（3×3+BN、1×1+BN、identity+BN），推理时合并成单个 3×3。

合并三步：① 每条分支各自 Conv-BN 融合 → ② 把 1×1 核零填充到 3×3 的**中心**、
identity 写成中心为 1 的对角核 → ③ 三个同形状的核直接相加。

> 零填充为什么成立：填进去的 0 乘任何值都是 0，**包括图像 padding 区的 0**，所以边界也等价。

In [ ]:
def pad_1x1_to_3x3(W1):
    '''(Cout,Cin,1,1) -> (Cout,Cin,3,3)，1x1 核放到 3x3 的中心。'''
    return np.pad(W1, ((0, 0), (0, 0), (1, 1), (1, 1)))

def identity_to_3x3(C):
    '''恒等映射写成 3x3 卷积核：中心位置的对角线为 1，其余为 0。'''
    K = np.zeros((C, C, 3, 3))
    for c in range(C):
        K[c, c, 1, 1] = 1.0
    return K

# 三条分支各自的参数（identity 分支要求 Cin==Cout 且 stride=1）
W1 = rng.normal(size=(C, C, 1, 1)) * 0.3
g1, be1, mu1, v1 = rng.normal(1, .2, C), rng.normal(0, .2, C), rng.normal(0, .3, C), rng.uniform(.5, 2., C)
gi, bei, mui, vi = rng.normal(1, .2, C), rng.normal(0, .2, C), rng.normal(0, .3, C), rng.uniform(.5, 2., C)
zero = np.zeros(C)

# —— 训练期：三分支相加 ——
y3 = bn_infer(conv2d(x, W3, None, pad=1), g3, be3, mu3, v3)
y1 = bn_infer(conv2d(x, W1, None, pad=0), g1, be1, mu1, v1)
yid = bn_infer(x, gi, bei, mui, vi)
y_multi = y3 + y1 + yid

# —— 推理期：合并成单个 3x3 ——
W3f, b3f = fuse_conv_bn(W3, zero, g3, be3, mu3, v3)                       # 步骤 ①
W1f, b1f = fuse_conv_bn(W1, zero, g1, be1, mu1, v1)
Wif, bif = fuse_conv_bn(identity_to_3x3(C), zero, gi, bei, mui, vi)
W_merged = W3f + pad_1x1_to_3x3(W1f) + Wif                                # 步骤 ②③
b_merged = b3f + b1f + bif
y_merged = conv2d(x, W_merged, b_merged, pad=1)

err = float(np.abs(y_multi - y_merged).max())
print('训练期三分支相加 -> %s' % (y_multi.shape,))
print('推理期单个 3x3   -> %s' % (y_merged.shape,))
print('最大绝对误差 %.3e' % err)
assert np.allclose(y_multi, y_merged, atol=1e-10), err
assert W_merged.shape == (C, C, 3, 3)
# 边界也必须等价（零填充区同样成立）—— 单独检查最外圈
assert np.allclose(y_multi[:, :, 0, :], y_merged[:, :, 0, :], atol=1e-10), '上边界不等价'
assert np.allclose(y_multi[:, :, :, -1], y_merged[:, :, :, -1], atol=1e-10), '右边界不等价'
print('\n✅ **三分支 = 单个 3x3，数值严格等价，边界也等价。**')
print('   训练时用多分支（多条梯度通路，好优化）；推理时用单分支（无中间张量，好部署）。')

In [ ]:
def rep_block_cost(C=256, H=40, W=40, dtype_bytes=2):
    '''训练期三分支 vs 推理期单 3x3 的算子/访存/计算账。'''
    hw = H * W
    macs3, macs1 = C * C * 9 * hw, C * C * 1 * hw
    bn_ops = 2 * C * hw                                   # BN 推理 = 一次乘一次加
    train_macs = macs3 + macs1 + 3 * bn_ops + 2 * C * hw  # 3 个 BN + 2 次逐元素加
    tensor = C * hw * dtype_bytes                         # 一份中间张量的字节数
    return {
        'train_macs': train_macs, 'infer_macs': macs3,
        'train_kernels': 3 + 3 + 2, 'infer_kernels': 1,   # 3 conv + 3 bn + 2 add
        'train_tensors': 3, 'infer_tensors': 0,
        'tensor_MB': tensor / 1e6,
    }

c = rep_block_cost()
print('RepVGG 块（C=256, 40x40 特征图, fp16）:')
print('%-26s %14s %14s' % ('', '训练期(三分支)', '推理期(单 3x3)'))
print('%-26s %14d %14d' % ('kernel 启动次数', c['train_kernels'], c['infer_kernels']))
print('%-26s %14d %14d' % ('中间张量份数', c['train_tensors'], c['infer_tensors']))
print('%-26s %13.3fG %13.3fG' % ('MACs', c['train_macs'] / 1e9, c['infer_macs'] / 1e9))
print('%-26s %13.2fMB %13.2fMB'
      % ('额外访存(中间张量)', c['train_tensors'] * c['tensor_MB'], 0.0))

assert c['infer_kernels'] == 1 and c['train_kernels'] == 8
assert c['infer_macs'] < c['train_macs']
ratio = c['infer_macs'] / c['train_macs']
assert 0.85 < ratio < 0.95, ratio
print('\n>>> MACs 只降了 %.0f%%，但 kernel 从 8 个变成 1 个、中间张量从 3 份变成 0 份。'
      % (100 * (1 - ratio)))
print('⚠️  **真正的收益不是 FLOPs 而是访存与 kernel 启动** —— 小模型上这才是瓶颈。')
print('⚠️  另一面：合并后的等效核数值分布更长尾（几条分支的极值叠加），')
print('    **INT8 per-tensor 量化会掉得比原生单分支更多** -> 用 per-channel 或 QAT（C60 模块 03）。')

## 5 · 解耦头 vs 耦合头：一笔必须会算的账

YOLOX 三件套里唯一有推理成本的一项。**代价的主项是两条支路各 2 个 3×3，
即 `4·c_mid²·9` 每像素** —— 所以 c_mid 是唯一的杠杆。

In [ ]:
def head_cost(c_in, c_mid, n_cls, hw_list, decoupled=True, n_anchor=1):
    '''返回 (params, macs)。多层 FPN **共享同一套头的权重**（RTMDet 式）：
       params 只算一次；macs 按各层空间尺寸求和。params 含 bias，macs 不含。
       耦合头 : 一个 1x1 conv  c_in -> n_anchor*(5+n_cls)
       解耦头 : 1x1 降维 c_in->c_mid；cls/reg 两条支路各 2 个 3x3；三个 1x1 输出头'''
    hw = sum(hw_list)
    if not decoupled:
        out = n_anchor * (5 + n_cls)
        return c_in * out + out, c_in * out * hw
    p, m = c_in * c_mid + c_mid, c_in * c_mid                 # stem 1x1
    for _ in range(4):                                        # 2 支路 x 2 个 3x3
        p += c_mid * c_mid * 9 + c_mid
        m += c_mid * c_mid * 9
    for co in (n_cls, 4, 1):                                  # cls / reg / obj
        p += c_mid * co + co
        m += c_mid * co
    return p, m * hw

LEVELS = [80 * 80, 40 * 40, 20 * 20]        # 640 输入的三层 FPN
CFGS = [('耦合头 (1x1, 3 anchor x 85)', dict(c_mid=256, decoupled=False, n_anchor=3)),
        ('解耦头 c_mid=256 (YOLOX 式)', dict(c_mid=256, decoupled=True)),
        ('解耦头 c_mid=64  (轻量化)',   dict(c_mid=64,  decoupled=True))]

base = None
print('%-30s %14s %14s %10s' % ('检测头', '参数量', 'MACs(三层)', '相对耦合头'))
out = {}
for name, kw in CFGS:
    p, m = head_cost(256, kw['c_mid'], 80, LEVELS,
                     decoupled=kw['decoupled'], n_anchor=kw.get('n_anchor', 1))
    out[name] = (p, m)
    if base is None:
        base = m
    print('%-30s %14s %13.2fG %9.1fx' % (name, '{:,}'.format(p), m / 1e9, m / base))

p_c, m_c = out['耦合头 (1x1, 3 anchor x 85)']
p_d, m_d = out['解耦头 c_mid=256 (YOLOX 式)']
p_l, m_l = out['解耦头 c_mid=64  (轻量化)']
assert (p_c, m_c) == (65535, 548352000), (p_c, m_c)
assert (p_d, m_d) == (2447957, 20551372800), (p_d, m_d)
assert m_d / m_c > 30, '解耦头(c_mid=256) 的 MACs 是耦合头的 30 倍以上'
assert m_l * 10 < m_d, 'c_mid 降到 1/4，代价降到约 1/14（3x3 项按 c_mid^2 走）'

print('\n>>> 解耦头(c_mid=256) 是耦合头的 %.1fx；把 c_mid 压到 64 只剩 %.1fx。'
      % (m_d / m_c, m_l / m_c))
print('>>> c_mid 降 4 倍 -> 代价降 %.1f 倍（主项 4*c_mid^2*9 按平方走）。' % (m_d / m_l))
print('\n✅ 所以后续工作不是不用解耦头，而是**把 c_mid 压到很小**：')
print('   YOLOv8 用 max(c_in/4, 16, reg_max*4)；RTMDet 用跨层共享权重摊薄参数量。')
print('⚠️  注意本函数按**共享权重**计。YOLOX 是每层独立头，params 要再乘层数。')

## 6 · 一对多 vs 一对一：YOLOv10 的一致双分配

复现两件事：
1. **不去重会怎样** —— 一对多分支必须靠 NMS 压回去，而 NMS 阈值极其敏感
2. **为什么必须"一致"** —— 两个头用不同度量时，o2o 的 top-1 常常不在 o2m 的 top-k 里

TaskAligned 度量：$t = s^{\alpha}\cdot u^{\beta}$，$(\alpha,\beta)=(0.5, 6)$。

In [ ]:
STRIDE, GRID = 8, 80                                  # 640 输入的 P3 层
gy, gx = np.mgrid[0:GRID, 0:GRID]
anchors = np.stack([(gx + 0.5) * STRIDE, (gy + 0.5) * STRIDE], -1).reshape(-1, 2).astype(float)

GPOS = [(80, 80), (220, 90), (360, 100), (500, 110), (120, 240), (280, 250),
        (440, 260), (560, 270), (100, 400), (260, 410), (420, 420), (560, 430)]
GSZ = [26, 20, 34, 18, 30, 22, 28, 24, 32, 20, 26, 36]
GTS = np.array([[p[0], p[1], s, s] for p, s in zip(GPOS, GSZ)], float)   # 12 个交通标志

def simulate_head(anchors, gts, seed=0, mis=0.35):
    '''模拟一个训练好的 dense head 的输出场：
       u = 定位质量（该 anchor 解码出的框与 GT 的 IoU）
       s = 分类分数（峰值**刻意偏离** GT 中心 —— cls-loc misalignment 是真实现象）'''
    r = np.random.default_rng(seed)
    s, u = np.full(len(anchors), 0.02), np.zeros(len(anchors))
    for (cx, cy, w, h) in gts:
        dx, dy = np.abs(anchors[:, 0] - cx), np.abs(anchors[:, 1] - cy)
        inter = np.clip(w - dx, 0, None) * np.clip(h - dy, 0, None)
        u = np.maximum(u, inter / (2 * w * h - inter))
        off = r.normal(0, mis * w, 2)
        d = np.hypot(anchors[:, 0] - (cx + off[0]), anchors[:, 1] - (cy + off[1]))
        s = np.maximum(s, 0.95 * np.exp(-(d ** 2) / (2 * (0.5 * w) ** 2)))
    return s, u

def per_gt_topk(metric, anchors, gts, k):
    '''中心先验：候选集限制在 GT 附近，再按 metric 取 top-k。'''
    picks = []
    for (cx, cy, w, h) in gts:
        inside = (np.abs(anchors[:, 0] - cx) <= w) & (np.abs(anchors[:, 1] - cy) <= h)
        idx = np.flatnonzero(inside)
        picks.append(idx[np.argsort(-metric[idx])][:k])
    return picks

ALPHA, BETA = 0.5, 6.0
s, u = simulate_head(anchors, GTS, seed=0)
t_align = (s ** ALPHA) * (u ** BETA)          # TaskAligned / v10 的一致度量
t_clsonly = s                                  # 「不一致」的对照：只看分类分数

o2m = per_gt_topk(t_align, anchors, GTS, k=10)      # 一对多：每 GT 10 个正样本
o2o = per_gt_topk(t_align, anchors, GTS, k=1)       # 一对一：每 GT 1 个
print('12 个交通标志，%d 个候选格点' % len(anchors))
print('   一对多分配的正样本总数: %d' % sum(len(p) for p in o2m))
print('   一对一分配的正样本总数: %d' % sum(len(p) for p in o2o))
assert sum(len(p) for p in o2m) == 120 and sum(len(p) for p in o2o) == 12
print('\nBETA=6 让 IoU 的权重远大于分类分数: 0.88^6=%.3f 而 0.70^6=%.3f（差 %.1f 倍）'
      % (0.88 ** 6, 0.70 ** 6, 0.88 ** 6 / 0.70 ** 6))

In [ ]:
def to_xyxy(b):
    return np.stack([b[..., 0] - b[..., 2] / 2, b[..., 1] - b[..., 3] / 2,
                     b[..., 0] + b[..., 2] / 2, b[..., 1] + b[..., 3] / 2], -1)

def iou_one_vs_many(a, B):
    a, B = to_xyxy(a), to_xyxy(B)
    x1, y1 = np.maximum(a[0], B[:, 0]), np.maximum(a[1], B[:, 1])
    x2, y2 = np.minimum(a[2], B[:, 2]), np.minimum(a[3], B[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ar = (a[2] - a[0]) * (a[3] - a[1])
    br = (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1])
    return inter / (ar + br - inter)

def nms(boxes, thr=0.6):
    order, keep = np.argsort(-boxes[:, 4]), []
    while len(order):
        i = order[0]; keep.append(int(i))
        if len(order) == 1:
            break
        rest = order[1:]
        order = rest[iou_one_vs_many(boxes[i], boxes[rest]) < thr]
    return np.array(keep)

def emit(picks, gts, seed=1, jitter=1.2):
    '''把正样本 anchor 解码成预测框（带一点回归误差）。'''
    r = np.random.default_rng(seed)
    out = []
    for p, (cx, cy, w, h) in zip(picks, gts):
        for _ in p:
            j = r.normal(0, jitter, 4)
            out.append([cx + j[0], cy + j[1], w + j[2], h + j[3], 0.90 - 0.001 * len(out)])
    return np.array(out)

raw_m, raw_o = emit(o2m, GTS), emit(o2o, GTS)
print('%-14s %10s %s' % ('NMS IoU 阈值', '一对多(120框)', '一对一(12框)'))
for thr in (0.30, 0.40, 0.45, 0.50, 0.60, 0.70):
    nm, no = len(nms(raw_m, thr)), len(nms(raw_o, thr))
    tag = '' if nm == 12 else '   <-- 漏删 %d 个重复' % (nm - 12)
    print('%-14.2f %10d %10d%s' % (thr, nm, no, tag))

assert len(raw_m) == 120 and len(raw_o) == 12
assert len(nms(raw_m, 0.45)) == 12
assert len(nms(raw_m, 0.60)) > 12, '常用默认阈值 0.6 就已经漏删重复框了'
assert all(len(nms(raw_o, t)) == 12 for t in (0.3, 0.45, 0.6, 0.7)), '一对一对阈值完全不敏感'

print('\n>>> 一对多必须靠 NMS 压回去，而 **NMS 阈值极其敏感**：0.45 刚好、0.6 漏删 3 个、0.7 漏删 6 个。')
print('>>> 一对一在任何阈值下都是 12 —— 因为它根本不需要 NMS。')
print('\n⚠️  阈值调低的另一面更危险：会把**真正相邻的两个目标合并成一个**。')
print('    而 TSR 恰好充满相邻目标 —— 门架上并排的限速牌、主辅牌组合、限速+解除牌。')
print('✅ 无 NMS 的第二个收益常被低估：**去掉了一个必须逐场景调、调不好两头出错的超参**。')

In [ ]:
# 「一致」到底有多重要：两个头用不同度量时会怎样
rates_align, rates_cls, same_top1 = [], [], []
for seed in range(20):                                   # 20 组随机的 cls-loc 失配
    s_, u_ = simulate_head(anchors, GTS, seed=seed)
    ta, tc = (s_ ** ALPHA) * (u_ ** BETA), s_
    m10 = per_gt_topk(ta, anchors, GTS, 10)              # o2m 分支（始终用对齐度量）
    a1 = per_gt_topk(ta, anchors, GTS, 1)                # o2o 用**同一个**度量
    c1 = per_gt_topk(tc, anchors, GTS, 1)                # o2o 用**纯分类分数**
    rates_align.append(np.mean([p[0] in set(q) for p, q in zip(a1, m10)]))
    rates_cls.append(np.mean([p[0] in set(q) for p, q in zip(c1, m10)]))
    same_top1.append(np.mean([p[0] == q[0] for p, q in zip(c1, m10)]))

ra, rc, st = float(np.mean(rates_align)), float(np.mean(rates_cls)), float(np.mean(same_top1))
print('12 个标志 x 20 组随机种子:')
print('%-42s %8s' % ('两个头的度量', 'o2o 的 top-1 落在 o2m top-10 内'))
print('%-42s %7.1f%%' % ('**一致**（都用 t = s^0.5 * u^6）', 100 * ra))
print('%-42s %7.1f%%' % ('不一致（o2o 改用纯分类分数）', 100 * rc))
print('\n更严格的指标 —— 两个头的 top-1 **完全相同**的比例: %.1f%%' % (100 * st))

assert ra == 1.0, '同一度量的 top-1 必然在自己的 top-10 里'
assert rc < 0.85, rc
assert st < 0.40, st
print('\n>>> 不一致时，o2o 选中的位置有 %.1f%% 根本不在 o2m 的 top-10 里 ——' % (100 * (1 - rc)))
print('    同一个位置被一个头当正样本、被另一个头当负样本，梯度互相抵消。')
print('>>> 而两个头 top-1 完全相同的比例只有 %.1f%%（分类分数最高的位置定位往往不准）。' % (100 * st))
print('\n✅ 这就是 YOLOv10 「**一致**双分配」里那个「一致」的含义：')
print('   不是「两个头都要有」，而是「两个头必须用同一个匹配度量」。')
print('✅ 和 DETR 一对一匹配的三个区别：① DETR 是全局二分图匹配(匈牙利)，v10 是逐 GT 取 top-1；')
print('   ② v10 有 o2m 头额外提供密集监督，DETR 没有（这是它 500 epoch 的根因之一，C54）；')
print('   ③ v10 的 o2o 头仍是**密集预测**，保留了 CNN 对小目标的全部优势。')

## ✏️ 练习 1：重参数化的两块积木

实现 `pad_1x1_to_3x3(W1)` 与 `identity_to_3x3(C)`：

- `pad_1x1_to_3x3`：把 `(Cout,Cin,1,1)` 的核零填充成 `(Cout,Cin,3,3)`，**1×1 的值放在中心**
- `identity_to_3x3`：返回 `(C,C,3,3)`，它作为 `pad=1` 的卷积核时等价于恒等映射

In [ ]:
def pad_1x1_to_3x3(W1):
    # TODO: (Cout,Cin,1,1) -> (Cout,Cin,3,3)，把 1x1 的值放到 3x3 的**中心**
    raise NotImplementedError

def identity_to_3x3(C):
    # TODO: (C,C,3,3)，中心位置 [c,c,1,1] = 1，其余全 0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Wp = pad_1x1_to_3x3(W1)
assert Wp.shape == (C, C, 3, 3), Wp.shape
assert np.allclose(Wp[:, :, 1, 1], W1[:, :, 0, 0]), '1x1 的值必须落在**中心**'
assert Wp[:, :, 0, 0].sum() == 0 and Wp[:, :, 2, 2].sum() == 0, '其余 8 个位置必须是 0'
# 零填充后用 pad=1 卷积，必须等价于原 1x1 用 pad=0 卷积（含边界）
assert np.allclose(conv2d(x, Wp, pad=1), conv2d(x, W1, pad=0), atol=1e-12)

K = identity_to_3x3(C)
assert K.shape == (C, C, 3, 3) and abs(K.sum() - C) < 1e-12, '只有 C 个 1'
assert K[0, 0, 1, 1] == 1.0 and K[0, 1, 1, 1] == 0.0 and K[0, 0, 0, 1] == 0.0
assert np.allclose(conv2d(x, K, pad=1), x, atol=1e-12), '它必须真的是恒等映射'

print('pad_1x1_to_3x3(W1)[0,0] =')
print(np.round(Wp[0, 0], 3))
print('\nidentity_to_3x3(3)[0,0] =')
print(identity_to_3x3(3)[0, 0])
print('\n✅ 练习 1 通过。零填充为什么成立：填进去的 0 乘任何值都是 0，')
print('   **包括图像 padding 区的 0** —— 所以边界也严格等价。')

## ✏️ 练习 2：把整个 RepVGG 块合并成一个 3×3

实现 `merge_rep_branches(W3, bn3, W1, bn1, bn_id, C)`，返回合并后的 `(W, b)`。

- `bn*` 是四元组 `(gamma, beta, mean, var)`；可直接用上面给好的 `fuse_conv_bn`
- **`bn_id` 可以是 `None`** —— 当 stride≠1 或输入输出通道数不同时，identity 分支不存在，
  这时只合并 3×3 与 1×1 两条分支。**这个边界条件是真实实现里必须处理的。**

In [ ]:
def merge_rep_branches(W3, bn3, W1, bn1, bn_id, C, eps=1e-5):
    # TODO: ① 三条分支各自 fuse_conv_bn（卷积分支的 bias 传 np.zeros(C)）
    #       ② 1x1 核 pad 到 3x3；identity 用 identity_to_3x3(C) 再过 fuse_conv_bn
    #       ③ 三个 3x3 核相加、bias 相加；bn_id 为 None 时跳过第三条分支
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
bn3, bn1_, bnid = (g3, be3, mu3, v3), (g1, be1, mu1, v1), (gi, bei, mui, vi)

Wm, bm = merge_rep_branches(W3, bn3, W1, bn1_, bnid, C)
assert Wm.shape == (C, C, 3, 3) and bm.shape == (C,), (Wm.shape, bm.shape)
err3 = float(np.abs(conv2d(x, Wm, bm, pad=1) - y_multi).max())
assert err3 < 1e-10, err3

# 无 identity 分支（stride!=1 或通道数变化时的真实情况）
Wm2, bm2 = merge_rep_branches(W3, bn3, W1, bn1_, None, C)
err2 = float(np.abs(conv2d(x, Wm2, bm2, pad=1) - (y3 + y1)).max())
assert err2 < 1e-10, err2

print('三分支合并  最大绝对误差 %.3e' % err3)
print('两分支合并  最大绝对误差 %.3e   （无 identity 分支）' % err2)
print('\n合并后等效核的数值分布 vs 单条 3x3 分支：')
for nm, W_ in [('单条 3x3 分支', W3), ('合并后的等效核', Wm)]:
    a = np.abs(W_)
    print('   %-16s  max/p90 = %6.2f   （越大越难 per-tensor 量化）'
          % (nm, a.max() / np.percentile(a, 90)))
print('\n✅ 练习 2 通过：**数学上严格等价**。')
print('⚠️  但注意最后两行 —— 合并后的核往往更长尾，这就是 INT8 掉点的来源（C60 模块 03）。')
print('⚠️  另外两个前提：① identity 要求 Cin==Cout 且 stride=1；')
print('   ② 合并必须在 BN 用 **running statistics**（eval 模式）之后做，否则权重是错的且不报错。')

## ✏️ 练习 3：anchor 拟合体检报告

实现 `anchor_fit_report(wh, anchors, thr=0.25)`，返回 dict：

| 字段 | 含义 |
|---|---|
| `mean_best_iou` | 每个框与最佳 anchor 的 IoU 的均值 |
| `bpr` | best possible recall：最佳 IoU ≥ `thr` 的比例 |
| `at_smallest` | 最佳 anchor 恰好是**面积最小**那个的比例 → **anchor 下界不够**的信号 |
| `at_largest` | 最佳 anchor 是**面积最大**那个的比例 → anchor 上界不够 |

**即使你用的是 anchor-free 检测器，这份体检也值得做**——它告诉你数据的尺度跨度
与现有 stride 够不够细。

In [ ]:
def anchor_fit_report(wh, anchors, thr=0.25):
    # TODO: 用 wh_iou 得到 (n,k) 矩阵，再算上表四个字段
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r9 = anchor_fit_report(wh, a9)
rc = anchor_fit_report(wh, COCO_ANCHORS)

assert abs(r9['mean_best_iou'] - mean_best_iou(wh, a9)) < 1e-12
assert r9['bpr'] == 1.0 and rc['bpr'] < 0.97, (r9['bpr'], rc['bpr'])
assert r9['mean_best_iou'] > rc['mean_best_iou'] + 0.2
assert rc['at_smallest'] > 0.5 > r9['at_smallest'], (rc['at_smallest'], r9['at_smallest'])
assert rc['at_largest'] < 0.01, 'TSR 里没有大目标，最大的 [373,326] 基本没人用'
assert abs(sum(anchor_fit_report(wh, a9[:1])[k] for k in ('at_smallest', 'at_largest')) - 2.0) < 1e-9, \
    '只有一个 anchor 时它既是最小也是最大'

print('%-22s %14s %10s %14s %13s' % ('anchor 来源', 'mean_best_iou', 'BPR', '挤在最小', '挤在最大'))
for nm, a in [('COCO 默认', COCO_ANCHORS), ('TSR 重新聚类 k=9', a9),
              ('TSR 重新聚类 k=3', kmeans_anchors(wh, 3))]:
    r = anchor_fit_report(wh, a)
    print('%-22s %14.4f %10.4f %13.1f%% %12.1f%%'
          % (nm, r['mean_best_iou'], r['bpr'], 100 * r['at_smallest'], 100 * r['at_largest']))

print('\n✅ 练习 3 通过。**怎么读这份报告**：')
print('   BPR < 0.98        -> 有样本永远匹配不上，必须重新聚类（v5 的 autoanchor 就用这条）')
print('   at_smallest 很高  -> anchor 尺寸下界不够 -> 加更小的 anchor 或加 P2 层')
print('   at_largest 很高   -> 上界不够 -> 加更大的 anchor 或加更粗的 stride')
print('   mean_best_iou 低  -> anchor 数量不够，或数据尺度跨度太大（考虑分层建模）')

## ✏️ 练习 4：检测头的参数量 / FLOPs 账

实现 `head_cost(c_in, c_mid, n_cls, hw_list, decoupled=True, n_anchor=1)`，
返回 `(params, macs)`。约定见第 5 节的 docstring：多层 FPN **共享同一套头的权重**，
`params` 只算一次且含 bias，`macs` 不含 bias 并按各层空间尺寸求和。

**这是面试里可能被要求当场算的一道题**——考的不是记忆，是你会不会拆一个结构的成本。

In [ ]:
def head_cost(c_in, c_mid, n_cls, hw_list, decoupled=True, n_anchor=1):
    # TODO: 耦合头 = 一个 1x1: c_in -> n_anchor*(5+n_cls)
    #       解耦头 = 1x1 降维 c_in->c_mid  +  4 个 3x3 (c_mid->c_mid)  +  三个 1x1 (n_cls/4/1)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert head_cost(256, 256, 80, LEVELS, decoupled=False, n_anchor=3) == (65535, 548352000)
assert head_cost(256, 256, 80, LEVELS, decoupled=True) == (2447957, 20551372800)
assert head_cost(256, 64, 80, LEVELS, decoupled=True) == (169685, 1421952000)
# 单层时 macs 应按空间尺寸线性缩放，params 不变
p1, m1 = head_cost(256, 64, 80, [6400], decoupled=True)
p3, m3 = head_cost(256, 64, 80, LEVELS, decoupled=True)
assert p1 == p3 and abs(m3 / m1 - sum(LEVELS) / 6400) < 1e-9
# c_mid 减半，3x3 主项降 4 倍
assert head_cost(256, 128, 80, LEVELS)[1] < head_cost(256, 256, 80, LEVELS)[1] / 3

print('%-10s %14s %14s %12s' % ('c_mid', '参数量', 'MACs(三层)', '相对耦合头'))
base = head_cost(256, 256, 80, LEVELS, decoupled=False, n_anchor=3)[1]
for cm in (256, 128, 64, 32, 16):
    p, m = head_cost(256, cm, 80, LEVELS, decoupled=True)
    print('%-10d %14s %13.2fG %11.1fx' % (cm, '{:,}'.format(p), m / 1e9, m / base))
print('\n✅ 练习 4 通过。**代价的主项是 4·c_mid²·9 每像素** —— c_mid 是唯一的杠杆。')
print('   在模块 00 的 2.45 ms 预算下，c_mid=256 的解耦头一项就吃掉了整个网络的算力预算。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def pad_1x1_to_3x3(W1):
    return np.pad(W1, ((0, 0), (0, 0), (1, 1), (1, 1)))

def identity_to_3x3(C):
    K = np.zeros((C, C, 3, 3))
    for c in range(C):
        K[c, c, 1, 1] = 1.0
    return K

In [ ]:
# 练习 2 参考答案
def merge_rep_branches(W3, bn3, W1, bn1, bn_id, C, eps=1e-5):
    zero = np.zeros(C)
    Wa, ba = fuse_conv_bn(W3, zero, *bn3, eps=eps)                       # 3x3 分支
    Wb, bb = fuse_conv_bn(W1, zero, *bn1, eps=eps)                       # 1x1 分支
    W, b = Wa + pad_1x1_to_3x3(Wb), ba + bb
    if bn_id is not None:                                                # identity 分支（可选）
        Wc, bc = fuse_conv_bn(identity_to_3x3(C), zero, *bn_id, eps=eps)
        W, b = W + Wc, b + bc
    return W, b

In [ ]:
# 练习 3 参考答案
def anchor_fit_report(wh, anchors, thr=0.25):
    m = wh_iou(wh, anchors)
    best, best_iou = m.argmax(1), m.max(1)
    area = anchors[:, 0] * anchors[:, 1]
    return {'mean_best_iou': float(best_iou.mean()),
            'bpr': float((best_iou >= thr).mean()),
            'at_smallest': float((best == int(np.argmin(area))).mean()),
            'at_largest': float((best == int(np.argmax(area))).mean())}

In [ ]:
# 练习 4 参考答案
def head_cost(c_in, c_mid, n_cls, hw_list, decoupled=True, n_anchor=1):
    hw = sum(hw_list)
    if not decoupled:
        out = n_anchor * (5 + n_cls)
        return c_in * out + out, c_in * out * hw
    p, m = c_in * c_mid + c_mid, c_in * c_mid              # 1x1 降维 stem
    for _ in range(4):                                     # cls/reg 两条支路各 2 个 3x3
        p += c_mid * c_mid * 9 + c_mid
        m += c_mid * c_mid * 9
    for co in (n_cls, 4, 1):                               # cls / reg / obj 三个 1x1
        p += c_mid * co + co
        m += c_mid * co
    return p, m * hw

---
## 🧪 真实工程胶囊：一份可直接抄走的「重参数化 + anchor + 无 NMS」清单

这三件事在真实项目里各有一个必须做的验证步骤，缺了就会变成上线后才发现的问题。

In [ ]:
RECIPE = r'''
# ============ ① 结构重参数化：融合 + **必须做的数值对拍** ============
model.eval()                                   # 关键：BN 必须用 running statistics
with torch.no_grad():
    x = torch.randn(2, 3, 640, 640)
    y_before = model(x)                        # 融合前（训练结构）
    model.fuse()                               # ultralytics: Conv+BN 融合 & RepBlock 合并
    y_after = model(x)                         # 融合后（部署结构）
torch.testing.assert_close(y_before, y_after, rtol=1e-4, atol=1e-5)
# ^ 这一行是不可省的。融合写错**不会报错**，只会表现为「导出后精度莫名下降」。
# 融合后再验一次 INT8：rep 块合并后权重分布更长尾，per-tensor 量化掉点会更明显
#   -> 优先 per-channel；掉点 >1-2 mAP 时对 rep 块做 QAT（C60 模块 03）

# ============ ② anchor：用**自己的数据**重新聚类，并读体检报告 ============
# ultralytics 训练时会自动跑 autoanchor；BPR < 0.98 时才重聚类。手动做法：
#   from ultralytics.utils.autoanchor import check_anchors, kmean_anchors
#   anchors = kmean_anchors(dataset, n=9, img_size=1280, thr=4.0, gen=1000)
# 必看三个数：BPR、mean_best_iou、挤在最小 anchor 的比例
#   BPR < 0.98        -> 有样本永远匹配不上
#   at_smallest 很高  -> anchor 下界不够 -> 加更小 anchor 或加 P2 层（TSR 的典型症状）
# anchor-free 检测器也要跑这个体检 —— 它体检的是**数据**，不是 anchor

# ============ ③ 无 NMS（YOLOv10）：导出时别把 NMS 又加回去 ============
#   yolo export model=yolov10s.pt format=engine half=True
# 检查点：ONNX 图里不应出现 NMS/TopK/NonMaxSuppression 节点；
#         输出应当是固定 shape (batch, 300, 6) 而不是动态的 nnz
# 无 NMS 买到的是**延迟的确定性**，不必然是延迟的均值 -> 用模块 00 的协议测 p99 再决定

# ============ ④ 改动归因：任何一项都要单独测 ============
# 单变量 / 同 epoch / 同增强 / 同分辨率 / 同调参预算 / 多种子
# 检测任务同配置不同种子的 mAP 波动典型 ±0.2-0.5 -> **+0.3 很可能是噪声**（C61 模块 01）
'''
print(RECIPE)
for key in ['model.eval()', 'assert_close', 'fuse()', 'BPR < 0.98',
            'at_smallest', 'p99', '单变量', '种子']:
    assert key in RECIPE, key
print('✅ 覆盖：融合对拍 / 量化风险 / anchor 体检 / 无 NMS 导出检查 / 归因纪律')

### 小结

- **YOLOv1 的失败是表达能力问题不是容量问题**：每格 B=2 + 一份类别向量，
  9 个标志结构上只能表达 6 个。加参数一条都解决不了 —— 必须改输出结构。
- **anchor 聚类的距离必须是 1−IoU**（欧氏距离被大框主导）。更重要的是
  **它是数据的体检报告**：COCO 默认 anchor 用在 TSR 上，4.1% 的标志永远匹配不上、
  60% 挤在最小的 anchor 上。**anchor-free 检测器也该跑这个体检。**
- **结构重参数化是严格数值等价的**（误差 2.7e-15），收益主要不是 FLOPs（只降 10%）
  而是 **kernel 从 8 个变 1 个、中间张量从 3 份变 0 份**。
  代价在别处：**合并后的核更长尾，INT8 会掉得更多。**
- **解耦头是 YOLOX 三件套里唯一收费的一项**：c_mid=256 时是耦合头的 37.5×。
  主项 `4·c_mid²·9` 按平方走，所以后续工作全都在压 c_mid。
  **采纳顺序：SimOTA（免费）→ anchor-free（省钱）→ 解耦头（最后，且压 c_mid）。**
- **YOLOv10 「一致」双分配的「一致」是指两个头用同一个匹配度量**。
  换成纯分类分数时，o2o 的 top-1 有 26% 落不进 o2m 的 top-10，两头 top-1 相同的只有 22%。
  无 NMS 买到的是**延迟的确定性**与**少调一个敏感超参**，不必然是延迟均值。
- **贯穿全模块的心法：凡是只在训练期付出代价的改动都是免费的午餐。**
  这条演进线上的大部分收益（分配、增强、辅助头、重参数化）都落在训练期 ——
  **所以被问「这几个点是从哪来的」，答案几乎总是「训练侧」，而不是「架构侧」。**

下一站：**模块 02 · 标签分配** —— 把本模块反复提到的那条主线完整走一遍。